[070] 소스 코드 Ⅱ-28 데이터 읽기: https://github.com/seongui2030/Section2/blob/main/Titanic.csv

In [1]:
import os
import urllib.request

# 1. ./data 폴더 생성 (이미 존재하면 무시)
os.makedirs("./data", exist_ok=True)

# 2. milknew.csv 파일을 ./data/milknew.csv 경로로 다운로드
url = "https://raw.githubusercontent.com/seongui2030/Section2/main/Titanic.csv"
urllib.request.urlretrieve(url, "./data/Titanic.csv")
print("다운로드 완료!")

다운로드 완료!


In [2]:
#  교재 Ⅱ-01 데이터 읽기
import pandas as pd
import numpy as np
df = pd.read_csv('/content/data/Titanic.csv')
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [3]:
#  교재 Ⅱ-02 데이터 속성 확인하기
df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')

[071] 데이터의 결측치 유무 확인하기

In [4]:
#  교재 Ⅱ-03 데이터 형태 확인하기
df.shape

(891, 12)

In [ ]:
#  교재 Ⅱ-04 결측치 유무 확인하기
df.isnull().sum()

In [6]:
#  교재 Ⅱ-06 핵심 속성 추출
df_core_columns = df.drop(['PassengerId', 'Name', 'Ticket'], axis=1)
df_core_columns.columns

Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Cabin',
       'Embarked'],
      dtype='object')

In [7]:
# 교재 Ⅱ-05 이상치 유무 파악하기

import numpy as np

# 1. IQR을 이용한 'Fare(요금)' 속성의 이상치 개수 확인 함수 정의
def outliers_iqr(dt, col):
    q1, q3 = np.percentile(dt[col], [25, 75])
    iqr = q3 - q1
    outlier_top = q3 + (1.5 * iqr)
    outlier_bottom = q1 - (1.5 * iqr)

    outliers_1 = dt[col] > outlier_top
    outliers_2 = dt[col] < outlier_bottom
    return dt[outliers_1 | outliers_2]

# 'Fare' 속성의 이상치 확인
fare_outliers = outliers_iqr(df_core_columns, 'Fare')
print(f"Fare 속성의 이상치 개수: {len(fare_outliers)}")

# 2. 결측치(누락된 값) 유무 파악
print("\n[속성별 결측치 개수]")
print(df_core_columns.isnull().sum())

Fare 속성의 이상치 개수: 116

[속성별 결측치 개수]
Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Cabin       687
Embarked      2
dtype: int64


[075] 데이터 전처리하기

In [8]:
# 교재 Ⅱ-07 결측치 제거하기(속성 제거)
df_notnull = df_core_columns.drop(['Cabin'], axis =1)
df.head(3)
# df.info()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [9]:
# 교재 Ⅱ-08 결측치 제거하기(데이터 제거)
df_notnull = df_notnull.dropna()
df_notnull.head(3)
# df_notnull.info()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S


### find_outliers = outliers_iqr(df_notnull, 'Fare')
- 타이타닉 데이터프레임(df_notnull)의 요금(Fare) 컬럼에서 IQR(사분위수 범위) 방식을 이용해 이상치를 찾아내는 함수를 실행하고, 그 결과를 find_outliers 변수에 저장하는 코드입니다.
### 이 코드가 내부적으로 작동하는 절차:
- 1. outliers_iqr(...) 함수 호출바로 윗셀([24번 셀])에서 정의해 둔 이상치 검출 함수를 불러옵니다.전달된 인자(입력값)는 처리 대상 데이터프레임인 df_notnull과 검사할 컬럼 이름인 'Fare'입니다.
- 2. 함수 내부의 IQR 계산 절차함수 안에서 다음과 같은 과정을 거쳐 이상치 기준을 세웁니다.Q1(1사분위수)과 Q3(3사분위수) 구하기: 요금 데이터를 크기순으로 정렬했을 때 25% 위치와 75% 위치의 값을 구합니다.
    - (np.percentile)IQR(Interquartile Range) 구하기: Q3 - Q1 값을 계산해 데이터의 중간 50%가 퍼져 있는 범위를 구합니다.
    - 상한선과 하한선 설정:상한선(Q3 + 1.5 * IQR): 이 금액보다 비싼 요금은 너무 높은 이상치로 판단합니다.- 하한선(Q1 - 1.5 * IQR): 이 금액보다 저렴한 요금은 너무 낮은 이상치로 판단합니다.
- 3. 이상치 데이터 추출 및 반환:
    - df_notnull['Fare']의 값이 위에서 구한 상한선보다 크거나 하한선보다 작은 이상치 행(Row)들만 골라냅니다. (dt[outliers_1 | outliers_2])그렇게 찾아낸 이상치 데이터 전체(행 데이터들)를 함수의 결과로 돌려줍니다.
- 4. find_outliers 변수에 저장함수의 리턴값(이상치에 해당하는 승객들의 행 데이터 목록)을 find_outliers라는 변수에 담아둡니다.이 변수에는 나중에 .index를 붙여서 "어떤 행들이 이상치였는지" 위치(인덱스) 번호만 쏙 빼내는 데 활용됩니다. (예: 바로 다음 줄의 df_notnull.loc[find_outliers.index, 'Fare'] = np.nan)

In [10]:
# 교재 Ⅱ-09 이상치가 있는 데이터를 평균값으로 치환해 보자
# 'Fare' 속성의 이상치를 결측치로 변경하고 확인해 보자
find_outliers = outliers_iqr(df_notnull, 'Fare') # IQR(사분위수 범위) 방식을 이용해 이상치를 찾아내는 함수를 실행하고, 그 결과를 find_outliers 변수에 저장
# print(find_outliers)
df_notnull.loc[find_outliers.index, 'Fare'] = np.nan # 이상치로 판별된 행의 Fare(요금) 값을 결측치(NaN)로 변경
print(df_notnull.head(3))
df_notnull.isnull().sum()
print(df_notnull.isnull().sum())

   Survived  Pclass     Sex   Age  SibSp  Parch   Fare Embarked
0         0       3    male  22.0      1      0  7.250        S
1         1       1  female  38.0      1      0    NaN        C
2         1       3  female  26.0      0      0  7.925        S
Survived     0
Pclass       0
Sex          0
Age          0
SibSp        0
Parch        0
Fare        95
Embarked     0
dtype: int64


In [12]:
# 교재 Ⅱ-10 'Fare' 속성의 평균값을 구하여, 결측치에 평균값을 입력하고 확인해 보자.
mean_Fare = df_notnull['Fare'].mean(axis = 0) # 각 열(Column)을 기준으로 행들을 아래로 훑어가며 평균을 구합니다.
# print(mean_Fare)

df_notnull['Fare'].fillna(df_notnull['Fare'].mean(), inplace=True)
df_notnull['Fare']
# print(df_notnull['Fare'].iloc[1])

19.026802755267422


In [13]:
# 교재 75, Ⅱ-11 데이터 저장하기
df_notnull.to_csv('./data/Titanic2.csv')

[094] 기계학습 알고리즘의 모델 구현하기

[095]  Ⅱ-28. 데이터 읽기

In [45]:
# Ⅱ-28. 데이터 읽기
import pandas as pd
import numpy as np
df_notnull = pd.read_csv('/content/data/Titanic2.csv')

In [46]:
# Ⅱ-30. 데이터 확인하기
print(df_notnull.head())
print(df_notnull.shape)
print(df_notnull.info())

   Unnamed: 0  Survived  Pclass     Sex   Age  SibSp  Parch       Fare  \
0           0         0       3    male  22.0      1      0   7.250000   
1           1         1       1  female  38.0      1      0  19.026803   
2           2         1       3  female  26.0      0      0   7.925000   
3           3         1       1  female  35.0      1      0  53.100000   
4           4         0       3    male  35.0      0      0   8.050000   

  Embarked  
0        S  
1        C  
2        S  
3        S  
4        S  
(712, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 712 entries, 0 to 711
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  712 non-null    int64  
 1   Survived    712 non-null    int64  
 2   Pclass      712 non-null    int64  
 3   Sex         712 non-null    object 
 4   Age         712 non-null    float64
 5   SibSp       712 non-null    int64  
 6   Parch       712 non-null    int

In [47]:
df_notnull.columns

Index(['Unnamed: 0', 'Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch',
       'Fare', 'Embarked'],
      dtype='object')

[095] 소스 코드 Ⅱ-30 데이터 전처리하기(데이터의 형태 변경)

In [48]:
#  범주형 데이터(Sex, Embarked 등) 인코딩 (문자 -> 숫자)
# df_notnull.info()
# df_notnull = pd.get_dummies(df_notnull, columns=['Sex'], drop_first=True) # 범주형 데이터를 원-핫 인코딩할때 첫 번째 열을 자동 삭제(성별은 Sex_male이 0 이면 무조건 Sex_female은 1이 되므로 하나의 열로 다른 하나의 열만 있어도 다른 열의 값을 완벽하게 유추 할수 있으므로 하나만 남김)
df_notnull = pd.get_dummies(df_notnull, columns=['Sex']) # 범주형 데이터를 원-핫 인코딩할때 첫 번째 열을 자동 삭제(성별은 Sex_male이 0 이면 무조건 Sex_female은 1이 되므로 하나의 열로 다른 하나의 열만 있어도 다른 열의 값을 완벽하게 유추 할수 있으므로 하나만 남김)
df_notnull = pd.get_dummies(df_notnull, columns=['Embarked'], prefix='town')

In [49]:
df_notnull

,Unnamed: 0,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_female,Sex_male,town_C,town_Q,town_S
0,0,0,3,22.0,1,0,7.250000,False,True,False,False,True
1,1,1,1,38.0,1,0,19.026803,True,False,True,False,False
2,2,1,3,26.0,0,0,7.925000,True,False,False,False,True
3,3,1,1,35.0,1,0,53.100000,True,False,False,False,True
4,4,0,3,35.0,0,0,8.050000,False,True,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...
707,885,0,3,39.0,0,5,29.125000,True,False,False,True,False
708,886,0,2,27.0,0,0,13.000000,False,True,False,False,True
709,887,1,1,19.0,0,0,30.000000,True,False,False,False,True
710,889,1,1,26.0,0,0,30.000000,False,True,True,False,False


In [50]:
# o_sex = pd.get_dummies(df_notnull.Sex)
# o_embarked = pd.get_dummies(df_notnull.Embarked)
# # print(df_notnull.head(3))
# df_notnull = pd.concat([df_notnull, o_sex, o_embarked], axis = 1)
df_notnull.shape

(712, 12)

[096] 소스 코드 Ⅱ-31 데이터 전처리하기(전처리한 데이터 확인)

In [51]:
df_notnull.head(3)

,Unnamed: 0,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_female,Sex_male,town_C,town_Q,town_S
0,0,0,3,22.0,1,0,7.250000,False,True,False,False,True
1,1,1,1,38.0,1,0,19.026803,True,False,True,False,False
2,2,1,3,26.0,0,0,7.925000,True,False,False,False,True


[096] 소스 코드 Ⅱ-32 데이터 전처리하기(불필요한 속성 제거)

In [ ]:
# df_notnull = df_notnull.drop(['Sex', 'Embarked'], axis = 1)
# df_notnull = df_notnull.drop(['Sex', 'Embarked'], axis = 1)
# df_notnull.head(3)

[097] 소스 코드 Ⅱ-33 데이터 전처리하기(데이터 표현 변경)

In [52]:
df_notnull['Fare'] = df_notnull['Fare'].round(2)
df_notnull.head(3)

,Unnamed: 0,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_female,Sex_male,town_C,town_Q,town_S
0,0,0,3,22.0,1,0,7.25,False,True,False,False,True
1,1,1,1,38.0,1,0,19.03,True,False,True,False,False
2,2,1,3,26.0,0,0,7.92,True,False,False,False,True


[097] 소스 코드 Ⅱ-34 데이터 전처리하기(종속 변수와 독립 변수 지정)

In [53]:
X = df_notnull.loc[:, 'Pclass':]
y = df_notnull['Survived']
# X.head()
# y.head()

[097] 소스 코드 Ⅱ-35 데이터 전처리하기(데이터 분할과 형태 확인)

In [54]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y)
print('train data 개수:', X_train.shape)
print('test data 개수:', X_test.shape)

train data 개수: (534, 10)
test data 개수: (178, 10)


[098] 소스 코드 Ⅱ-36 모델 생성하기(K-최근접 이웃)

In [55]:
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors = 5)
knn.fit(X_train, y_train)

KNeighborsClassifier()

[098] 소스 코드 Ⅱ-37 모델의 성능 평가하기

In [56]:
from sklearn import metrics
y_pred = knn.predict(X_test)
acc = metrics.accuracy_score(y_test, y_pred)
acc

0.6573033707865169

[098] 소스 코드 Ⅱ-38 최적의 K 값 찾기

In [57]:
k = 10
acc_array = np.zeros(k)
for k in np.arange(1,k+1,1):
    knn = KNeighborsClassifier(n_neighbors = k).fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    acc = metrics.accuracy_score(y_test, y_pred)
    acc_array[k-1] = acc
    print('k: ', k, '정확도: ', acc)

k:  1 정확도:  0.6910112359550562
k:  2 정확도:  0.6573033707865169
k:  3 정확도:  0.6629213483146067
k:  4 정확도:  0.6910112359550562
k:  5 정확도:  0.6573033707865169
k:  6 정확도:  0.6404494382022472
k:  7 정확도:  0.6966292134831461
k:  8 정확도:  0.6797752808988764
k:  9 정확도:  0.6853932584269663
k:  10 정확도:  0.6797752808988764


[099] 소스 코드 Ⅱ-39 모델 활용하기

In [58]:
df_notnull.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 712 entries, 0 to 711
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  712 non-null    int64  
 1   Survived    712 non-null    int64  
 2   Pclass      712 non-null    int64  
 3   Age         712 non-null    float64
 4   SibSp       712 non-null    int64  
 5   Parch       712 non-null    int64  
 6   Fare        712 non-null    float64
 7   Sex_female  712 non-null    bool   
 8   Sex_male    712 non-null    bool   
 9   town_C      712 non-null    bool   
 10  town_Q      712 non-null    bool   
 11  town_S      712 non-null    bool   
dtypes: bool(5), float64(2), int64(5)
memory usage: 42.5 KB


In [60]:
new_data1 = np.array([[3, 22.0, 1, 0, 52.1000, 0, 1, 1, 0, 0]])
new_data1 = pd.DataFrame(new_data1, columns = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_female', 'Sex_male', 'town_C', 'town_Q', 'town_S'])
prediction = knn.predict(new_data1)
print('생존 여부', prediction)

생존 여부 [1]


[099] 소스 코드 Ⅱ-40 모델 활용하기

In [61]:
new_data2 = np.array([[3, 55.0, 1, 0, 7.1000, 1, 0, 1, 0, 0]])
new_data2 = pd.DataFrame(new_data2, columns = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_female', 'Sex_male', 'town_C', 'town_Q', 'town_S'])
prediction = knn.predict(new_data2)
print('생존 여부', prediction)

생존 여부 [0]
